# Notebook 3: Data Quality Assessment

#### Overview
Investigate dataset integrity across three core data quality pillars: Missing Values, Duplicate Records, and Invalid/Anomalous Values.


In [1]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Customer_Data.csv")
df.head()

,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,InternetService,PaymentMethod,SeniorCitizen,SupportCalls,StreamingService,PaperlessBilling,Churn
0,CUST00001,56.0,Female,0,20.41,131.92,One year,DSL,Bank transfer,No,3,No,Yes,Yes
1,CUST00002,69.0,Male,9,21.12,394.45,Two year,DSL,Credit card,Yes,1,Yes,No,No
2,CUST00003,46.0,Female,3,61.66,2755.45,Two year,DSL,Bank transfer,No,3,Yes,No,No
3,CUST00004,32.0,Male,8,35.53,796.58,Month-to-month,No,Credit card,No,2,No,No,No
4,CUST00005,60.0,Male,9,86.45,10188.80,Two year,Fiber optic,Electronic check,Yes,3,No,Yes,Yes


### 1. Missing Values Analysis
Evaluating total count, percentage, and column-wise distribution of missing values across the dataset.

In [2]:
# Count missing values per column
missing_count = df.isnull().sum()

# Calculate missing value percentages
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Combine into a summary dataframe
missing_summary = pd.DataFrame(
    {"Missing Count": missing_count, "Missing Percentage (%)": missing_percentage}
)

print("--- Missing Values by Column ---")
print(missing_summary)

--- Missing Values by Column ---
                  Missing Count  Missing Percentage (%)
CustomerID                    0                     0.0
Age                           0                     0.0
Gender                        0                     0.0
TenureYears                   0                     0.0
MonthlyCharges                0                     0.0
TotalCharges                  0                     0.0
ContractType                  0                     0.0
InternetService               0                     0.0
PaymentMethod                 0                     0.0
SeniorCitizen                 0                     0.0
SupportCalls                  0                     0.0
StreamingService              0                     0.0
PaperlessBilling              0                     0.0
Churn                         0                     0.0


### 2. Missing Value Patterns
Checking for co-occurrence patterns or structural missingness across features.

In [3]:
# Check overall missingness in dataset
total_missing = df.isnull().sum().sum()
print(f"Total Missing Values across entire dataset: {total_missing}")

# Display rows containing any null values (if any exist)
null_rows = df[df.isnull().any(axis=1)]
print(f"Number of rows with missing values: {len(null_rows)}")
null_rows.head()

Total Missing Values across entire dataset: 0
Number of rows with missing values: 0


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,InternetService,PaymentMethod,SeniorCitizen,SupportCalls,StreamingService,PaperlessBilling,Churn


### 3. Duplicate Records
Identifying exact row duplicates and potential subset duplicate records.

In [4]:
# Count completely duplicated rows
exact_duplicates = df.duplicated().sum()
print(f"Total Completely Duplicated Rows: {exact_duplicates}")

# Check for duplicate CustomerIDs (potential key duplication)
id_duplicates = df.duplicated(subset=["CustomerID"]).sum()
print(f"Duplicate CustomerID Records: {id_duplicates}")

Total Completely Duplicated Rows: 0
Duplicate CustomerID Records: 0


### 4. Invalid Values & Logical Anomalies
Checking numerical bounds, categorical consistency, null-strings, and data types.

In [5]:
# 1. Check for negative values in numeric columns where inappropriate
numeric_cols = df.select_dtypes(include=["number"]).columns
negative_counts = (df[numeric_cols] < 0).sum()
print("--- Negative Values Count ---")
print(negative_counts)


--- Negative Values Count ---
Age               0
TenureYears       0
MonthlyCharges    0
TotalCharges      0
SupportCalls      0
dtype: int64


In [6]:
# 2. Check categorical columns for empty strings or null-like text ("None", "N/A", "null", " ")
cat_cols = df.select_dtypes(include=["object"]).columns
null_string_patterns = ["", " ", "null", "none", "n/a", "na", "?"]

print("--- Null-like Strings in Categorical Columns ---")
for col in cat_cols:
    invalid_mask = df[col].astype(str).str.strip().str.lower().isin(null_string_patterns)
    print(f"{col}: {invalid_mask.sum()} invalid entries")

--- Null-like Strings in Categorical Columns ---
CustomerID: 0 invalid entries
Gender: 0 invalid entries
ContractType: 0 invalid entries
InternetService: 0 invalid entries
PaymentMethod: 0 invalid entries
SeniorCitizen: 0 invalid entries
StreamingService: 0 invalid entries
PaperlessBilling: 0 invalid entries
Churn: 0 invalid entries


C:\Users\abarn\AppData\Local\Temp\ipykernel_3720\3564463675.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns


In [7]:
# 3. Inspect unique values for categorical attributes to spot incorrect categories
print("--- Unique Values per Categorical Feature ---")
for col in cat_cols:
    print(f"\nColumn: {col}")
    print(df[col].unique())

--- Unique Values per Categorical Feature ---

Column: CustomerID
<ArrowStringArray>
['CUST00001', 'CUST00002', 'CUST00003', 'CUST00004', 'CUST00005', 'CUST00006',
 'CUST00007', 'CUST00008', 'CUST00009', 'CUST00010',
 ...
 'CUST05991', 'CUST05992', 'CUST05993', 'CUST05994', 'CUST05995', 'CUST05996',
 'CUST05997', 'CUST05998', 'CUST05999', 'CUST06000']
Length: 6000, dtype: str

Column: Gender
<ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str

Column: ContractType
<ArrowStringArray>
['One year', 'Two year', 'Month-to-month']
Length: 3, dtype: str

Column: InternetService
<ArrowStringArray>
['DSL', 'No', 'Fiber optic']
Length: 3, dtype: str

Column: PaymentMethod
<ArrowStringArray>
['Bank transfer', 'Credit card', 'Electronic check']
Length: 3, dtype: str

Column: SeniorCitizen
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

Column: StreamingService
<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

Column: PaperlessBilling
<ArrowStringArray>
['Yes', 'No']
Length: 2

### 5. Summary Report: Data Quality Findings

| Assessment Area | Findings / Status | Recommendation |
| :--- | :--- | :--- |
| **Missing Values** | 0 missing values across all columns | No imputation required |
| **Duplicate Records** | 0 exact duplicate rows found | Primary keys are unique |
| **Invalid / Negative Values** | All numeric ranges (`Age`, `TenureYears`, `MonthlyCharges`) are valid ($\ge 0$) | No range trimming needed |
| **Categorical Consistency** | Valid distinct string values without missing-string artifacts | Ready for encoding |